In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS catalog.silver;

In [0]:
%sql
SELECT * FROM catalog.bronze.patients_raw LIMIT 10;

patient_id,first_name,last_name,gender,dob,city,philhealth_no,contact_number,blood_type,date_registered,assigned_branch,_rescued_data,_source_file,_ingest_timestamp,ingestion_date
P0001,Gabriel,Castillo,MALE,11/24/1990,Iloilo City,PH9963334018,9553035110,B-,2015-01-03,H009,null,abfss://data@datastoragea.dfs.core.windows.net/staging/patients_raw/patients_raw,2026-05-05T11:35:56.318Z,2026-05-05
P0002,Sofia,Delos Santos,F,2007-03-23,Mabalacat,PH6756332150,9266944844,O-,2018-06-04,H002,null,abfss://data@datastoragea.dfs.core.windows.net/staging/patients_raw/patients_raw,2026-05-05T11:35:56.318Z,2026-05-05
P0003,Kenneth,Torres,M,20-11-1967,San Fernando,PH3479708607,9856528252,B-,2016-01-22,H005,null,abfss://data@datastoragea.dfs.core.windows.net/staging/patients_raw/patients_raw,2026-05-05T11:35:56.318Z,2026-05-05
P0004,Isabella,Co,F,1972-11-09,Iligan,PH3783290795,9754049436,B-,2017-09-24,H008,null,abfss://data@datastoragea.dfs.core.windows.net/staging/patients_raw/patients_raw,2026-05-05T11:35:56.318Z,2026-05-05
P0005,Juan,Lim,M,23-10-1972,Tanauan,PH7439149233,9790256940,AB+,2022-03-09,H009,null,abfss://data@datastoragea.dfs.core.windows.net/staging/patients_raw/patients_raw,2026-05-05T11:35:56.318Z,2026-05-05
P0006,Cristina,Legaspi,F,1982-02-05,Tagum,PH8217611860,9740389325,A+,2016-07-13,H008,null,abfss://data@datastoragea.dfs.core.windows.net/staging/patients_raw/patients_raw,2026-05-05T11:35:56.318Z,2026-05-05
P0007,Kenneth,Delos Santos,M,1958-03-15,Manila,PH4919706735,9771410971,B+,2019-11-17,H003,null,abfss://data@datastoragea.dfs.core.windows.net/staging/patients_raw/patients_raw,2026-05-05T11:35:56.318Z,2026-05-05
P0008,Jerome,Reyes,M,1993-05-08,Quezon City,PH1338258951,9885879795,null,2022-02-25,H003,null,abfss://data@datastoragea.dfs.core.windows.net/staging/patients_raw/patients_raw,2026-05-05T11:35:56.318Z,2026-05-05
P0009,Joshua,Chan,M,1980-12-23,Las Pinas,PH8357059814,9528414718,O-,2020-08-17,H004,null,abfss://data@datastoragea.dfs.core.windows.net/staging/patients_raw/patients_raw,2026-05-05T11:35:56.318Z,2026-05-05
P0010,Jose,Ocampo,M,06/03/1983,Lapu-Lapu,PH6317189421,9818309417,B-,2022-04-18,H010,null,abfss://data@datastoragea.dfs.core.windows.net/staging/patients_raw/patients_raw,2026-05-05T11:35:56.318Z,2026-05-05


In [0]:
from pyspark.sql.functions import sha2, col, concat_ws, monotonically_increasing_id, current_timestamp, when, upper, coalesce, try_to_date
from delta.tables import DeltaTable


In [0]:
bronze_table = 'catalog.bronze.patients_raw'
silver_table = 'catalog.silver.dim_patients'
checkpoint_path = "abfss://data@datastoragea.dfs.core.windows.net/silver/dim_patients/checkpoint/"

df_patients_bronze = spark.readStream.table(bronze_table)

df_patients_clean = (
    df_patients_bronze
        .drop(
            "philhealth_no",
            "contact_number",
            "blood_type",
            "_rescued_data",
            "_source_file",
            "_ingest_timestamp",
            "ingestion_date"
        )
        .withColumn(
            "gender",
            when(upper(col("gender")).isin("MALE", "M"), "M")
            .when(upper(col("gender")).isin("FEMALE", "F"), "F")
            .otherwise(None)
            )
        .withColumn(
            "dob", coalesce(
                try_to_date(col("dob"), "yyyy-MM-dd"),
                try_to_date(col("dob"), "MM/dd/yyyy"),
                try_to_date(col("dob"), "dd-MM-yyyy"),
                try_to_date(col("dob"), "MMMM dd yyyy"),   # handles "November 27 1971"
            )
        )
        .withColumn("patient_name_hash", sha2(concat_ws('|', col("first_name"), col("last_name")), 256))
        .withColumn("load_timestamp", current_timestamp())
)

def merge_dim_patients(batch_df, batch_id):
    batch_df = batch_df.dropDuplicates(["patient_id"])

    if not spark.catalog.tableExists(silver_table):
        # First run — create the table
        (batch_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(silver_table))  
        return
    
    # Load Delta table by name and upsert into Silver
    dim_patients = DeltaTable.forName(spark, silver_table)

    (dim_patients.alias("t")
        .merge(
            batch_df.alias("s"),
            "t.patient_id = s.patient_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    

In [0]:
(   
    df_patients_clean.writeStream
        .foreachBatch(merge_dim_patients)  # for every batch, merge_dim_patients is run
        .outputMode("update")
        .trigger(availableNow=True)
        .option("checkpointLocation", checkpoint_path)
        .start()
)

In [0]:
%sql
SELECT * FROM catalog.silver.dim_patients LIMIT 10;

patient_id,first_name,last_name,gender,dob,city,date_registered,assigned_branch,patient_name_hash,load_timestamp
P0179,Christian,Aquino,M,1972-04-14,Iligan,2019-05-08,H007,47a2de2a99379f79c9a9a5b766c7d7ea5b75e535d9337baae70266858e5c1efc,2026-05-22T15:37:35.596Z
P0243,Veronica,Morales,F,1964-08-22,Cebu City,2016-05-03,H004,6615e4dc33b6be458341cd540677c11626de0f519a9d910e01d75ec1d29ea908,2026-05-22T15:37:35.596Z
P0092,Rowena,Navarro,F,1977-02-15,Paranaque,2019-05-28,H003,10356abbb26147f88b3dae4b734b10cf661cf9ef9951c015e23110c3a212a476,2026-05-22T15:37:35.596Z
P0248,Corazon,Gonzales,F,1970-05-16,Davao City,2022-07-07,H006,6efd3d28b035475dae7d43cbe22e9b6550779190ffb2a39ab4e7e54cbf133583,2026-05-22T15:37:35.596Z
P0136,Patricia,Soriano,F,1968-07-12,Taguig,2018-08-26,H002,a98c260da90f5b02c23250f38e56e980debd2ae0b99f7c4ac1d204593245511d,2026-05-22T15:37:35.596Z
P0023,Joaquin,Tan,M,1967-02-24,Lipa,2021-06-13,H009,424c3de676296288e7dc5afd55b5255e05ec159cbf4b3d010c54dd82632a34e8,2026-05-22T15:37:35.596Z
P0033,Elijah,Torres,M,1987-02-19,Manila,2021-07-23,H010,6b93002ab305f9861bdf3bfb5c0370d569b7b704aa570ef06fee8d82c15e4726,2026-05-22T15:37:35.596Z
P0105,Angelica,Padilla,F,1962-12-22,Tagum,2019-09-10,H005,80fb841c3683e334a5ef7527bb128c766953c8bc5516c3c98d1e5cc261c8a063,2026-05-22T15:37:35.596Z
P0156,Joaquin,Lim,M,1949-07-12,Iloilo City,2022-02-01,H001,1ed280a9eb34cf0ad264626f8801b90a78d08242f98e08d43bd301240dd4c438,2026-05-22T15:37:35.596Z
P0275,Maribel,Tolentino,F,1959-07-12,San Fernando,2015-05-06,H010,8c298e42733dc5b198d1fc2a41778fed345b8a0e1242b1e4eda1388c23517330,2026-05-22T15:37:35.596Z


In [0]:
# Data quality check
from pyspark.sql.functions import col, count, min, max, year, current_date, datediff

print("DIM PATIENTS")
df = spark.read.table("catalog.silver.dim_patients")
total = df.count()
print(f"Total rows: {total} (expected: 500)")

dup_id = total - df.select("patient_id").distinct().count()
print(f"Duplicate patient_ids: {'OK' if dup_id == 0 else dup_id}")

for c in ["patient_id", "gender", "dob", "assigned_branch"]:
    n = df.filter(col(c).isNull()).count()
    print(f"  {'OK' if n == 0 else 'CHECK'} {c}: {n} nulls")

# Gender should only be M or F after standardization
invalid_gender = df.filter(~col("gender").isin("M", "F")).count()
print(f"Invalid gender values: {'OK' if invalid_gender == 0 else invalid_gender}")

print("Gender distribution:")
df.groupBy("gender").count().show()

# Check DOB 
df = df.withColumn("age", (datediff(current_date(), col("dob")) / 365).cast("int"))
df.select(
    min("age").alias("min_age"),
    max("age").alias("max_age")
).show()

# Assigned branch should all be valid: H001-H010
invalid_branch = df.filter(
    ~col("assigned_branch").isin([f"H{i:03d}" for i in range(1,11)])
).count()
print(f"Invalid assigned_branch: {'OK' if invalid_branch == 0 else f'CHECK — {invalid_branch} rows'}")

DIM PATIENTS
Total rows: 500 (expected: 500)
Duplicate patient_ids: OK
  OK patient_id: 0 nulls
  OK gender: 0 nulls
  OK dob: 0 nulls
  OK assigned_branch: 0 nulls
Invalid gender values: OK
Gender distribution:
+------+-----+
|gender|count|
+------+-----+
|     M|  255|
|     F|  245|
+------+-----+

+-------+-------+
|min_age|max_age|
+-------+-------+
|     18|     83|
+-------+-------+

Invalid assigned_branch: CHECK — 14 rows
